# Scat Feature Extraction Demo: MFCC, Spectral Flux, ZCR, ContentVec, Causal VAD

This Colab-style notebook analyzes a monophonic voice/scat input with 10 ms frame features. It is intended as the first front-end prototype for the scat analyzer before mapping syllables such as `ddang`, `ddaeng`, or `ddeung` into Bass-DDSP articulation controls.

Outputs:
- waveform and log spectrogram
- MFCC and delta-MFCC heatmaps
- spectral flux and zero-crossing-rate curves
- causal Silero VAD probability from a streaming 32 ms chunk loop
- optional ContentVec embedding diagnostics if a local checkpoint is configured
- onset candidates from spectral flux
- frame feature tensors shaped `(T, C)` and `(T, C_plus)`


In [1]:
# Colab / local setup
import importlib
import os
import subprocess
import sys
import warnings


def ensure(package, import_name=None):
    import_name = import_name or package
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])


ensure('librosa')
ensure('soundfile')
ensure('numpy')
ensure('matplotlib')
ensure('scipy')
ensure('pandas')
ensure('ipywidgets')

from pathlib import Path
import io
import math
import numpy as np
import pandas as pd
import librosa
import librosa.display
import soundfile as sf
from scipy.signal import find_peaks
import matplotlib.pyplot as plt
from IPython.display import Audio, display

try:
    import torch
except ImportError as exc:
    raise ImportError('This notebook needs PyTorch for Silero VAD and optional ContentVec.') from exc

plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.22
np.set_printoptions(precision=4, suppress=True)

workspace_candidates = [Path('/workspace'), Path.cwd(), Path.cwd().parent]
WORKSPACE_ROOT = next(
    (candidate for candidate in workspace_candidates if (candidate / 'learn').exists() or (candidate / 'contentvec').exists()),
    Path.cwd(),
)
CONTENTVEC_REPO = Path(os.environ.get('CONTENTVEC_REPO', WORKSPACE_ROOT / 'contentvec')).expanduser()
SILERO_VAD_REPO = Path(os.environ.get('SILERO_VAD_REPO', WORKSPACE_ROOT / 'silero-vad')).expanduser()
_checkpoint_env = os.environ.get('CONTENTVEC_CHECKPOINT', '').strip()
CONTENTVEC_CHECKPOINT = Path(_checkpoint_env).expanduser() if _checkpoint_env else None

print('workspace root:', WORKSPACE_ROOT)
print('contentvec repo:', CONTENTVEC_REPO)
print('silero-vad repo:', SILERO_VAD_REPO)
print('CONTENTVEC_CHECKPOINT:', CONTENTVEC_CHECKPOINT or '(not set)')


workspace root: /workspace
contentvec repo: /workspace/contentvec
silero-vad repo: /workspace/silero-vad
CONTENTVEC_CHECKPOINT: (not set)


## 1. Choose Or Upload A Voice Input

This notebook is intended for the local SSH/dev-container workflow, not Google Colab.

Recommended workflow:
1. Record a short scat/voice file on your local machine or phone.
2. Put it under `/workspace/learn/voice_inputs/` in the container, or set `AUDIO_PATH` to the exact file path.
3. Run the loader cell. If `AUDIO_PATH` is empty, it automatically chooses the newest supported audio file in `voice_inputs`.

Direct microphone recording from Python is not reliable through SSH/dev containers because the kernel runs inside the remote container, not on your laptop. The optional uploader widget below works in some VS Code/Jupyter frontends; if it does not appear, manually copy or drag the file into `voice_inputs`.


### Optional Local Upload Widget

Run this cell if your VS Code/Jupyter frontend supports `ipywidgets.FileUpload`. Choose a local audio file, press **Save uploaded audio**, then run the loader cell below. If the widget does not work in your frontend, skip it and manually put files in `/workspace/learn/voice_inputs/`.


In [ ]:
INPUT_DIR = WORKSPACE_ROOT / 'learn' / 'voice_inputs'
INPUT_DIR.mkdir(parents=True, exist_ok=True)
SUPPORTED_AUDIO_EXTENSIONS = ('.wav', '.flac', '.mp3', '.ogg', '.m4a', '.aac')


def _iter_widget_uploads(value):
    if isinstance(value, dict):
        return list(value.values())
    if isinstance(value, (tuple, list)):
        return list(value)
    return []


try:
    import ipywidgets as widgets

    audio_uploader = widgets.FileUpload(
        accept=','.join(SUPPORTED_AUDIO_EXTENSIONS),
        multiple=False,
        description='Choose audio',
    )
    save_upload_button = widgets.Button(
        description='Save uploaded audio',
        button_style='primary',
    )
    upload_status = widgets.Output()

    def save_uploaded_audio(_=None):
        with upload_status:
            upload_status.clear_output()
            items = _iter_widget_uploads(audio_uploader.value)
            if not items:
                print('No file selected yet.')
                return
            item = items[0]
            name = item.get('name', 'uploaded_audio.wav')
            content = item.get('content', b'')
            if isinstance(content, memoryview):
                content = content.tobytes()
            out_path = INPUT_DIR / Path(name).name
            out_path.write_bytes(bytes(content))
            globals()['AUDIO_PATH'] = str(out_path)
            print('Saved:', out_path)
            print('AUDIO_PATH is now set for the loader cell.')

    save_upload_button.on_click(save_uploaded_audio)
    display(widgets.VBox([audio_uploader, save_upload_button, upload_status]))
except Exception as exc:
    print('ipywidgets upload UI is unavailable:', type(exc).__name__, exc)
    print('Manual path: copy audio files into', INPUT_DIR)


In [3]:
TARGET_SR = 16000

# Set this to an exact path when you want deterministic input selection.
# Example: AUDIO_PATH = '/workspace/learn/voice_inputs/ddang_take_01.wav'
AUDIO_PATH = globals().get('AUDIO_PATH', '')

INPUT_DIR = globals().get('INPUT_DIR', WORKSPACE_ROOT / 'learn' / 'voice_inputs')
INPUT_DIR = Path(INPUT_DIR)
INPUT_DIR.mkdir(parents=True, exist_ok=True)
SUPPORTED_AUDIO_EXTENSIONS = globals().get(
    'SUPPORTED_AUDIO_EXTENSIONS',
    ('.wav', '.flac', '.mp3', '.ogg', '.m4a', '.aac'),
)


def synthesize_fallback_scat(sr=TARGET_SR):
    rng = np.random.default_rng(7)
    syllables = []
    f0s = [105.0, 138.0, 92.0, 164.0]
    durations = [0.42, 0.36, 0.50, 0.44]
    for i, (f0, dur) in enumerate(zip(f0s, durations)):
        n = int(round(dur * sr))
        t = np.arange(n) / sr
        onset_noise = rng.normal(0, 1, n) * np.exp(-t / 0.018)
        vowel_env = (1.0 - np.exp(-t / 0.035)) * np.exp(-t / 0.65)
        vibrato = 0.012 * np.sin(2 * np.pi * 5.5 * t + i)
        phase = 2 * np.pi * np.cumsum(f0 * (1 + vibrato)) / sr
        harmonic = np.sin(phase) + 0.35 * np.sin(2 * phase + 0.4) + 0.16 * np.sin(3 * phase + 1.1)
        syllable = 0.65 * harmonic * vowel_env + 0.20 * onset_noise
        syllables.append(syllable.astype(np.float32))
        syllables.append(np.zeros(int(0.06 * sr), dtype=np.float32))
    y = np.concatenate(syllables)
    y = y / max(np.max(np.abs(y)), 1e-7) * 0.9
    return y.astype(np.float32), sr, 'synthetic_fallback_scat.wav'


def list_audio_inputs(input_dir=INPUT_DIR):
    files = []
    for ext in SUPPORTED_AUDIO_EXTENSIONS:
        files.extend(input_dir.rglob(f'*{ext}'))
        files.extend(input_dir.rglob(f'*{ext.upper()}'))
    return sorted(set(files), key=lambda p: p.stat().st_mtime, reverse=True)


def choose_audio_path():
    explicit = str(AUDIO_PATH).strip() if AUDIO_PATH is not None else ''
    if explicit:
        return Path(explicit).expanduser(), 'explicit AUDIO_PATH'
    candidates = list_audio_inputs(INPUT_DIR)
    if candidates:
        return candidates[0], 'newest file in voice_inputs'
    return None, 'fallback synthetic signal'


def load_local_or_fallback():
    chosen_path, reason = choose_audio_path()
    if chosen_path is not None:
        if not chosen_path.exists():
            raise FileNotFoundError(f'AUDIO_PATH does not exist: {chosen_path}')
        y, sr = librosa.load(chosen_path, sr=TARGET_SR, mono=True)
        return y.astype(np.float32), sr, str(chosen_path), reason
    return (*synthesize_fallback_scat(TARGET_SR), reason)


available_inputs = list_audio_inputs(INPUT_DIR)
print('input directory:', INPUT_DIR)
print('supported extensions:', SUPPORTED_AUDIO_EXTENSIONS)
print('available audio files:', len(available_inputs))
for candidate in available_inputs[:8]:
    print('  ', candidate)
if len(available_inputs) > 8:
    print('  ...')

y, sr, source_name, source_reason = load_local_or_fallback()
y, _ = librosa.effects.trim(y, top_db=45)
y = y.astype(np.float32)
y = y / max(np.max(np.abs(y)), 1e-7) * 0.95

print('source:', source_name)
print('selection:', source_reason)
print('sample rate:', sr)
print('duration:', len(y) / sr, 'seconds')
display(Audio(y, rate=sr))


input directory: /workspace/learn/voice_inputs
supported extensions: ('.wav', '.flac', '.mp3', '.ogg', '.m4a', '.aac')
available audio files: 1
   /workspace/learn/voice_inputs/Scat 1.m4a


/tmp/ipykernel_108963/2299635633.py:60: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(chosen_path, sr=TARGET_SR, mono=True)
/usr/local/lib/python3.10/dist-packages/librosa/core/audio.py:183: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


NoBackendError: 

## 2. Extract 10 ms Frame Features

`hop_length = 10 ms` matches the intended scat analyzer control frame rate. `frame_length = 25 ms` is conventional for short-time voice features.

In [4]:
HOP_SECONDS = 0.010
FRAME_SECONDS = 0.025
N_MFCC = 20

hop_length = int(round(HOP_SECONDS * sr))
win_length = int(round(FRAME_SECONDS * sr))
n_fft = 1
while n_fft < win_length:
    n_fft *= 2

stft = librosa.stft(y, n_fft=n_fft, hop_length=hop_length, win_length=win_length, center=True)
mag = np.abs(stft).astype(np.float32)
power = mag ** 2
times = librosa.frames_to_time(np.arange(mag.shape[1]), sr=sr, hop_length=hop_length)

mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC, n_fft=n_fft, hop_length=hop_length, win_length=win_length)
mfcc_delta = librosa.feature.delta(mfcc)
zcr = librosa.feature.zero_crossing_rate(y, frame_length=win_length, hop_length=hop_length, center=True)[0]

# Spectral flux: positive frame-to-frame spectral change after L1 magnitude normalization.
mag_norm = mag / np.maximum(np.sum(mag, axis=0, keepdims=True), 1e-8)
positive_diff = np.maximum(np.diff(mag_norm, axis=1), 0.0)
spectral_flux = np.concatenate([[0.0], np.sqrt(np.sum(positive_diff ** 2, axis=0))]).astype(np.float32)

def robust_normalize(x):
    x = np.asarray(x, dtype=np.float32)
    lo, hi = np.percentile(x, [5, 95])
    return np.clip((x - lo) / max(hi - lo, 1e-7), 0.0, 1.0)

flux_norm = robust_normalize(spectral_flux)
zcr_norm = robust_normalize(zcr)
frame_energy = librosa.feature.rms(S=mag, frame_length=n_fft, hop_length=hop_length, center=True)[0]
energy_norm = robust_normalize(frame_energy)

feature_tensor = np.concatenate([
    mfcc.T,
    mfcc_delta.T,
    zcr[:, None],
    spectral_flux[:, None],
    energy_norm[:, None],
], axis=1).astype(np.float32)

feature_names = (
    [f'mfcc_{i:02d}' for i in range(N_MFCC)]
    + [f'delta_mfcc_{i:02d}' for i in range(N_MFCC)]
    + ['zcr', 'spectral_flux', 'energy_norm']
)

print('hop_length:', hop_length, 'samples')
print('win_length:', win_length, 'samples')
print('n_fft:', n_fft)
print('mfcc:', mfcc.shape, '(coefficients, frames)')
print('zcr:', zcr.shape)
print('spectral_flux:', spectral_flux.shape)
print('feature_tensor:', feature_tensor.shape, '(T, C)')

NameError: name 'sr' is not defined

## 3. Detect Candidate Vocal Transients

These are not final labels. They are visual markers for consonant-like events where a scat classifier may reset or change articulation state.

In [ ]:
min_distance_frames = max(1, int(round(0.080 / HOP_SECONDS)))
peaks, properties = find_peaks(
    flux_norm,
    height=0.35,
    prominence=0.10,
    distance=min_distance_frames,
)
peak_times = times[peaks]
print('candidate transient frames:', peaks.tolist())
print('candidate transient times:', np.round(peak_times, 3).tolist())

## 4. ContentVec And Causal VAD

Silero VAD is run causally: the notebook feeds 512-sample chunks at 16 kHz, preserves the model state, and timestamps each probability at the moment it becomes available. ContentVec is optional because the cloned repository contains source code but no pretrained checkpoint; set `CONTENTVEC_CHECKPOINT=/path/to/checkpoint_best_legacy.pt` before launching the notebook, or edit the variable in the setup cell.


In [ ]:
VAD_THRESHOLD = 0.50
SILERO_CHUNK_SAMPLES = 512  # 32 ms at 16 kHz; Silero's causal streaming chunk size.
CONTENTVEC_FRAME_SECONDS = 0.020  # HuBERT/ContentVec convolutional stride: 320 samples at 16 kHz.
CONTENTVEC_LAYER = int(os.environ.get('CONTENTVEC_LAYER', '12'))
CONTENTVEC_SPK_EMB_DIM = int(os.environ.get('CONTENTVEC_SPK_EMB_DIM', '256'))


def _interp_to_frame_times(values, value_times, target_times, fill=0.0):
    values = np.asarray(values, dtype=np.float32)
    value_times = np.asarray(value_times, dtype=np.float32)
    target_times = np.asarray(target_times, dtype=np.float32)
    if values.size == 0 or value_times.size == 0 or target_times.size == 0:
        return np.full(target_times.shape, fill, dtype=np.float32)
    return np.interp(
        target_times,
        value_times,
        values,
        left=float(values[0]),
        right=float(values[-1]),
    ).astype(np.float32)


def _pca_numpy(x, n_components=2):
    x = np.asarray(x, dtype=np.float32)
    if x.ndim != 2 or x.shape[0] < 2 or x.shape[1] < 1:
        return np.zeros((x.shape[0] if x.ndim == 2 else 0, n_components), dtype=np.float32)
    centered = x - np.mean(x, axis=0, keepdims=True)
    _, _, vt = np.linalg.svd(centered, full_matrices=False)
    n = min(n_components, vt.shape[0])
    projected = centered @ vt[:n].T
    if n < n_components:
        projected = np.pad(projected, ((0, 0), (0, n_components - n)))
    return projected.astype(np.float32)


def run_causal_silero_vad(audio, sr):
    if sr != 16000:
        raise ValueError('Silero VAD cell expects TARGET_SR == 16000.')
    jit_path = SILERO_VAD_REPO / 'src' / 'silero_vad' / 'data' / 'silero_vad.jit'
    if not jit_path.exists():
        return np.zeros(0, dtype=np.float32), np.zeros(0, dtype=np.float32), f'missing Silero JIT file: {jit_path}'

    model = torch.jit.load(str(jit_path), map_location='cpu')
    model.eval()
    if hasattr(model, 'reset_states'):
        model.reset_states()

    wav = torch.from_numpy(np.asarray(audio, dtype=np.float32))
    pad = (-len(wav)) % SILERO_CHUNK_SAMPLES
    if pad:
        wav = torch.nn.functional.pad(wav, (0, pad))

    probs = []
    available_times = []
    with torch.no_grad():
        for start in range(0, wav.numel(), SILERO_CHUNK_SAMPLES):
            chunk = wav[start:start + SILERO_CHUNK_SAMPLES]
            prob = float(model(chunk, sr).reshape(-1)[0].item())
            probs.append(prob)
            available_times.append((start + SILERO_CHUNK_SAMPLES) / sr)

    return (
        np.asarray(probs, dtype=np.float32),
        np.asarray(available_times, dtype=np.float32),
        f'loaded {jit_path}',
    )


causal_vad_probs, causal_vad_times, causal_vad_status = run_causal_silero_vad(y, sr)
vad_prob_10ms = _interp_to_frame_times(causal_vad_probs, causal_vad_times, times, fill=0.0)
vad_speech_10ms = (vad_prob_10ms >= VAD_THRESHOLD).astype(np.float32)

print('causal VAD:', causal_vad_status)
print('causal VAD chunks:', len(causal_vad_probs), 'chunk_seconds:', SILERO_CHUNK_SAMPLES / sr)
print('causal VAD probability range:', (float(np.min(causal_vad_probs)), float(np.max(causal_vad_probs))) if causal_vad_probs.size else '(empty)')


def _find_contentvec_checkpoint():
    candidates = []
    if CONTENTVEC_CHECKPOINT is not None:
        candidates.append(CONTENTVEC_CHECKPOINT)
    candidates.extend([
        WORKSPACE_ROOT / 'checkpoints' / 'contentvec' / 'checkpoint_best_legacy.pt',
        WORKSPACE_ROOT / 'checkpoints' / 'contentvec' / 'checkpoint_best.pt',
        CONTENTVEC_REPO / 'checkpoints' / 'checkpoint_best_legacy.pt',
        CONTENTVEC_REPO / 'checkpoints' / 'checkpoint_best.pt',
    ])
    for candidate in candidates:
        if candidate is not None and candidate.exists():
            return candidate
    return None


def _extract_contentvec_features(audio, sr):
    ckpt = _find_contentvec_checkpoint()
    if ckpt is None:
        return None, None, 'skipped: no ContentVec checkpoint found. Set CONTENTVEC_CHECKPOINT to checkpoint_best_legacy.pt or checkpoint_best.pt.'
    if not CONTENTVEC_REPO.exists():
        return None, None, f'skipped: missing ContentVec repo at {CONTENTVEC_REPO}'

    if str(CONTENTVEC_REPO) not in sys.path:
        sys.path.insert(0, str(CONTENTVEC_REPO))

    try:
        import fairseq
    except Exception as exc:
        return None, None, f'skipped: fairseq is not importable ({type(exc).__name__}: {exc})'

    try:
        # Registers ContentVec model/task classes for fairseq checkpoint loading.
        import contentvec.models.hubert.contentvec  # noqa: F401
        import contentvec.tasks.contentvec_pretraining  # noqa: F401
    except Exception as exc:
        warnings.warn(f'ContentVec class registration warning: {type(exc).__name__}: {exc}')

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    try:
        models, cfg, task = fairseq.checkpoint_utils.load_model_ensemble_and_task([str(ckpt)])
        model = models[0].to(device).eval()
        wav = torch.from_numpy(np.asarray(audio, dtype=np.float32)).unsqueeze(0).to(device)
        padding_mask = torch.zeros(wav.shape, dtype=torch.bool, device=device)

        with torch.no_grad():
            try:
                out = model.extract_features(
                    source=wav,
                    padding_mask=padding_mask,
                    mask=False,
                    output_layer=CONTENTVEC_LAYER,
                )
            except TypeError:
                spk_emb = torch.zeros(1, CONTENTVEC_SPK_EMB_DIM, device=device)
                try:
                    out = model.extract_features(
                        wav,
                        spk_emb,
                        padding_mask=padding_mask,
                        mask=False,
                        output_layer=CONTENTVEC_LAYER,
                    )
                except TypeError:
                    out = model.extract_features(
                        source=wav,
                        spk_emb=spk_emb,
                        padding_mask=padding_mask,
                        mask=False,
                        output_layer=CONTENTVEC_LAYER,
                    )

        if isinstance(out, (tuple, list)):
            feats = out[0]
        elif isinstance(out, dict):
            feats = out.get('x', out.get('features'))
        else:
            feats = out
        if feats is None:
            return None, None, 'skipped: ContentVec returned no feature tensor.'
        feats = feats.squeeze(0).detach().cpu().numpy().astype(np.float32)
        cv_times = np.arange(feats.shape[0], dtype=np.float32) * CONTENTVEC_FRAME_SECONDS
        return feats, cv_times, f'loaded {ckpt} on {device}'
    except Exception as exc:
        return None, None, f'skipped: ContentVec extraction failed ({type(exc).__name__}: {exc})'


contentvec_features, contentvec_times, contentvec_status = _extract_contentvec_features(y, sr)
if contentvec_features is not None:
    contentvec_pca = _pca_numpy(contentvec_features, 2)
    contentvec_novelty = np.concatenate([
        [0.0],
        np.linalg.norm(np.diff(contentvec_features, axis=0), axis=1),
    ]).astype(np.float32)
    contentvec_novelty_norm = robust_normalize(contentvec_novelty)
    contentvec_novelty_10ms = _interp_to_frame_times(contentvec_novelty_norm, contentvec_times, times, fill=0.0)
    contentvec_pca_10ms = np.stack([
        _interp_to_frame_times(contentvec_pca[:, 0], contentvec_times, times, fill=0.0),
        _interp_to_frame_times(contentvec_pca[:, 1], contentvec_times, times, fill=0.0),
    ], axis=1).astype(np.float32)
else:
    contentvec_pca = np.zeros((0, 2), dtype=np.float32)
    contentvec_novelty = np.zeros(0, dtype=np.float32)
    contentvec_novelty_norm = np.zeros(0, dtype=np.float32)
    contentvec_novelty_10ms = np.zeros_like(times, dtype=np.float32)
    contentvec_pca_10ms = np.zeros((len(times), 2), dtype=np.float32)

print('ContentVec:', contentvec_status)
if contentvec_features is not None:
    print('ContentVec features:', contentvec_features.shape, '(frames, channels)')
    print('ContentVec frame seconds:', CONTENTVEC_FRAME_SECONDS)


## 5. Rich Visual Dashboard


In [ ]:
audio_time = np.arange(len(y)) / sr
log_power_db = librosa.power_to_db(power, ref=np.max)
mel_db = librosa.power_to_db(
    librosa.feature.melspectrogram(y=y, sr=sr, n_fft=n_fft, hop_length=hop_length, win_length=win_length, n_mels=96),
    ref=np.max,
)

fig = plt.figure(figsize=(16, 24), constrained_layout=True)
gs = fig.add_gridspec(10, 2, height_ratios=[1.0, 1.25, 1.25, 1.15, 1.05, 1.05, 1.25, 1.1, 1.1, 1.15])

ax = fig.add_subplot(gs[0, :])
ax.plot(audio_time, y, color='black', linewidth=0.7)
for t in peak_times:
    ax.axvline(t, color='#d62728', alpha=0.75, linewidth=1.0)
if causal_vad_probs.size:
    speech_mask = vad_speech_10ms > 0.5
    ax.fill_between(times, -1.0, 1.0, where=speech_mask, color='#4c78a8', alpha=0.12, step='mid', label='VAD speech')
ax.set_title('Waveform with spectral-flux transient candidates and causal VAD speech mask')
ax.set_ylabel('amplitude')
ax.set_xlim(0, audio_time[-1] if len(audio_time) else 0)
ax.legend(loc='upper right')

ax = fig.add_subplot(gs[1, :])
librosa.display.specshow(mel_db, sr=sr, hop_length=hop_length, x_axis='time', y_axis='mel', cmap='magma', ax=ax)
for t in peak_times:
    ax.axvline(t, color='cyan', alpha=0.75, linewidth=0.8)
ax.set_title('Log-mel spectrogram')

ax = fig.add_subplot(gs[2, :])
librosa.display.specshow(log_power_db, sr=sr, hop_length=hop_length, x_axis='time', y_axis='hz', cmap='magma', ax=ax)
ax.set_ylim(0, min(5000, sr / 2))
for t in peak_times:
    ax.axvline(t, color='cyan', alpha=0.75, linewidth=0.8)
ax.set_title('Linear-frequency STFT magnitude, dB')

ax = fig.add_subplot(gs[3, 0])
librosa.display.specshow(mfcc, x_axis='time', sr=sr, hop_length=hop_length, cmap='coolwarm', ax=ax)
ax.set_title('MFCC')
ax.set_ylabel('coefficient')

ax = fig.add_subplot(gs[3, 1])
librosa.display.specshow(mfcc_delta, x_axis='time', sr=sr, hop_length=hop_length, cmap='coolwarm', ax=ax)
ax.set_title('Delta MFCC')
ax.set_ylabel('coefficient')

ax = fig.add_subplot(gs[4, :])
ax.plot(times, flux_norm, color='#d62728', label='spectral flux, normalized', linewidth=1.4)
ax.plot(times, zcr_norm, color='#1f77b4', label='ZCR, normalized', linewidth=1.2)
ax.plot(times, energy_norm, color='#2ca02c', label='RMS energy, normalized', linewidth=1.0, alpha=0.85)
ax.scatter(peak_times, flux_norm[peaks], color='#d62728', s=32, zorder=3, label='flux peaks')
ax.set_title('Frame features for consonant/transient and noisy/voiced cues')
ax.set_xlabel('time (s)')
ax.set_ylabel('normalized value')
ax.set_xlim(0, times[-1] if len(times) else 0)
ax.legend(loc='upper right')

ax = fig.add_subplot(gs[5, :])
if causal_vad_probs.size:
    ax.step(causal_vad_times, causal_vad_probs, where='post', color='#4c78a8', linewidth=1.4, label='causal Silero VAD prob, 32 ms chunks')
    ax.plot(times, vad_prob_10ms, color='#f58518', linewidth=1.0, alpha=0.85, label='VAD prob interpolated to 10 ms')
    ax.axhline(VAD_THRESHOLD, color='black', linestyle='--', linewidth=0.9, label=f'threshold {VAD_THRESHOLD:.2f}')
    ax.fill_between(times, 0.0, 1.0, where=vad_speech_10ms > 0.5, color='#4c78a8', alpha=0.10, step='mid')
else:
    ax.text(0.01, 0.5, causal_vad_status, transform=ax.transAxes, va='center')
ax.set_title('Causal voice activity probability')
ax.set_xlabel('time when probability becomes available (s)')
ax.set_ylabel('speech probability')
ax.set_ylim(-0.02, 1.02)
ax.set_xlim(0, times[-1] if len(times) else 0)
ax.legend(loc='upper right')

ax = fig.add_subplot(gs[6, :])
if contentvec_features is not None:
    n_show = min(48, contentvec_features.shape[1])
    cv_show = contentvec_features[:, :n_show].T
    im = ax.imshow(
        cv_show,
        aspect='auto',
        interpolation='nearest',
        cmap='viridis',
        extent=[contentvec_times[0], contentvec_times[-1] if len(contentvec_times) else 0, n_show, 0],
    )
    ax.set_title(f'ContentVec embedding heatmap, first {n_show} channels')
    ax.set_ylabel('channel')
    ax.set_xlabel('time (s)')
    fig.colorbar(im, ax=ax, label='activation')
else:
    ax.text(0.01, 0.55, contentvec_status, transform=ax.transAxes, va='center', wrap=True)
    ax.set_title('ContentVec embedding heatmap')
    ax.set_axis_off()

ax = fig.add_subplot(gs[7, 0])
if contentvec_features is not None and len(contentvec_pca):
    sc = ax.scatter(contentvec_pca[:, 0], contentvec_pca[:, 1], c=contentvec_times, cmap='plasma', s=16, alpha=0.85)
    ax.set_title('ContentVec PCA trajectory')
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    fig.colorbar(sc, ax=ax, label='time (s)')
else:
    ax.text(0.05, 0.5, 'ContentVec PCA unavailable', transform=ax.transAxes, va='center')
    ax.set_axis_off()

ax = fig.add_subplot(gs[7, 1])
if contentvec_features is not None and len(contentvec_novelty_norm):
    ax.plot(contentvec_times, contentvec_novelty_norm, color='#b279a2', linewidth=1.2, label='ContentVec novelty')
    ax.plot(times, flux_norm, color='#d62728', alpha=0.45, linewidth=1.0, label='spectral flux')
    ax.set_xlim(0, times[-1] if len(times) else 0)
    ax.set_ylim(-0.02, 1.02)
    ax.legend(loc='upper right')
else:
    ax.text(0.05, 0.5, 'ContentVec novelty unavailable', transform=ax.transAxes, va='center')
    ax.set_axis_off()
ax.set_title('Content representation change vs acoustic flux')
ax.set_xlabel('time (s)')

ax = fig.add_subplot(gs[8, 0])
sc = ax.scatter(zcr, spectral_flux, c=times[:len(zcr)], cmap='viridis', s=18, alpha=0.85)
ax.set_title('Feature phase plot: ZCR vs spectral flux')
ax.set_xlabel('ZCR')
ax.set_ylabel('spectral flux')
fig.colorbar(sc, ax=ax, label='time (s)')

ax = fig.add_subplot(gs[8, 1])
ax.hist(zcr, bins=40, alpha=0.65, label='ZCR')
ax.hist(spectral_flux, bins=40, alpha=0.65, label='spectral flux')
if causal_vad_probs.size:
    ax.hist(causal_vad_probs, bins=40, alpha=0.50, label='VAD prob')
ax.set_title('Feature distributions')
ax.legend()

ax = fig.add_subplot(gs[9, :])
compact_rows = [
    robust_normalize(mfcc[0]),
    robust_normalize(mfcc[1]),
    robust_normalize(mfcc[2]),
    flux_norm,
    zcr_norm,
    energy_norm,
    vad_prob_10ms,
]
compact_labels = ['MFCC0', 'MFCC1', 'MFCC2', 'Flux', 'ZCR', 'Energy', 'VAD']
if contentvec_features is not None:
    compact_rows.append(contentvec_novelty_10ms)
    compact_labels.append('ContentVec novelty')
compact = np.vstack(compact_rows)
im = ax.imshow(compact, aspect='auto', interpolation='nearest', cmap='viridis', extent=[times[0], times[-1], compact.shape[0], 0])
ax.set_yticks(np.arange(0.5, len(compact_labels) + 0.5))
ax.set_yticklabels(compact_labels)
ax.set_xlabel('time (s)')
ax.set_title('Compact normalized feature/control candidate matrix')
fig.colorbar(im, ax=ax, label='normalized')

plt.show()


## 6. Export `(T, C)` Features

`feature_tensor` contains the baseline 10 ms acoustic features. `feature_tensor_plus` appends causal VAD probability and, when available, low-dimensional ContentVec diagnostics aligned to the same 10 ms frame grid.


In [ ]:
optional_blocks = []
optional_names = []

optional_blocks.append(vad_prob_10ms[:, None].astype(np.float32))
optional_names.append('causal_vad_prob')
optional_blocks.append(vad_speech_10ms[:, None].astype(np.float32))
optional_names.append('causal_vad_speech')

if contentvec_features is not None:
    optional_blocks.append(contentvec_pca_10ms.astype(np.float32))
    optional_names.extend(['contentvec_pca_0', 'contentvec_pca_1'])
    optional_blocks.append(contentvec_novelty_10ms[:, None].astype(np.float32))
    optional_names.append('contentvec_novelty_norm')

feature_tensor_plus = np.concatenate([feature_tensor] + optional_blocks, axis=1).astype(np.float32)
feature_names_plus = feature_names + optional_names

df = pd.DataFrame(feature_tensor_plus, columns=feature_names_plus)
df.insert(0, 'time_seconds', times[:len(df)])
df['flux_norm'] = flux_norm[:len(df)]
df['zcr_norm'] = zcr_norm[:len(df)]
df['energy_norm'] = energy_norm[:len(df)]
df['is_flux_peak'] = False
df.loc[peaks[peaks < len(df)], 'is_flux_peak'] = True

contentvec_features_np = (
    np.asarray(contentvec_features, dtype=np.float32)
    if contentvec_features is not None
    else np.zeros((0, 0), dtype=np.float32)
)
contentvec_times_np = (
    np.asarray(contentvec_times, dtype=np.float32)
    if contentvec_times is not None
    else np.zeros(0, dtype=np.float32)
)

out_npz = 'scat_features_10ms.npz'
out_csv = 'scat_features_10ms.csv'
np.savez(
    out_npz,
    y=y,
    sr=sr,
    times=times,
    feature_tensor=feature_tensor,
    feature_names=np.array(feature_names),
    feature_tensor_plus=feature_tensor_plus,
    feature_names_plus=np.array(feature_names_plus),
    mfcc=mfcc,
    mfcc_delta=mfcc_delta,
    zcr=zcr,
    spectral_flux=spectral_flux,
    flux_peaks=peaks,
    causal_vad_times=causal_vad_times,
    causal_vad_probs=causal_vad_probs,
    vad_prob_10ms=vad_prob_10ms,
    vad_speech_10ms=vad_speech_10ms,
    contentvec_features=contentvec_features_np,
    contentvec_times=contentvec_times_np,
    contentvec_pca=contentvec_pca,
    contentvec_pca_10ms=contentvec_pca_10ms,
    contentvec_novelty=contentvec_novelty,
    contentvec_novelty_10ms=contentvec_novelty_10ms,
    contentvec_status=np.array(contentvec_status),
    causal_vad_status=np.array(causal_vad_status),
)
df.to_csv(out_csv, index=False)

print('Saved:', out_npz)
print('Saved:', out_csv)
print('Base classifier input shape after batching:', (1, *feature_tensor.shape))
print('Plus classifier input shape after batching:', (1, *feature_tensor_plus.shape))
print('Optional added features:', optional_names)
display(df.head(12))

try:
    from google.colab import files
    print('Use files.download(out_npz) or files.download(out_csv) if you want local copies.')
except Exception:
    pass


## Notes For The Scat Analyzer

- MFCCs mainly represent vocal tract/timbre shape, useful for vowel-like states.
- Spectral flux emphasizes sudden spectral changes, useful for consonants and syllable onsets.
- ZCR is high for noisy/unvoiced consonants and breathy attacks.
- Causal VAD is useful as a speech/activity gate, not as a note-on detector by itself.
- ContentVec is useful for inspecting phonetic/content state changes, but it needs a pretrained checkpoint and usually runs at about 20 ms frame stride.
- The first practical classifier target should be framewise or segmentwise syllable/articulation probabilities, not direct waveform synthesis.
- For real-time use, keep the same hop size and use causal feature extraction or a small lookahead buffer.
